In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers, optimizers
import os
import warnings

warnings.filterwarnings('ignore')
keras.mixed_precision.set_global_policy('float32')
np.random.seed(42)
tf.random.set_seed(42)

data = pd.read_csv('../data/sp500_dataset.csv', index_col='Date', parse_dates=True)

data['Target'] = data['Close'].shift(-1)
data = data.dropna()

features = ['Open', 'High', 'Low', 'Close', 'Volume', 
            'SMA_50', 'SMA_200', 'MACD', 'Signal', 'RSI', 'Momentum']

X = data[features].values
y = data['Target'].values
dates = data.index.to_numpy()

LOOKBACK_WINDOW = 20

def create_sequences_with_dates(X, y, dates, window_size):
    X_seq, y_seq, y_dates = [], [], []
    for i in range(len(X) - window_size):
        X_seq.append(X[i:(i + window_size)])
        y_seq.append(y[i + window_size])
        y_dates.append(dates[i + window_size])
    return (np.array(X_seq, dtype=np.float32),
            np.array(y_seq, dtype=np.float32),
            np.array(y_dates, dtype='datetime64[ns]'))

X_seq, y_seq, dates_seq = create_sequences_with_dates(X, y, dates, LOOKBACK_WINDOW)

TEST_SIZE = 0.20
test_split_index = int(len(X_seq) * (1 - TEST_SIZE))
X_train_raw, X_test_raw = X_seq[:test_split_index], X_seq[test_split_index:]
y_train_raw, y_test_raw = y_seq[:test_split_index], y_seq[test_split_index:]
dates_test = dates_seq[test_split_index:]

scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))

n_features = X_train_raw.shape[2]
X_train_2d = X_train_raw.reshape(-1, n_features)
X_test_2d = X_test_raw.reshape(-1, n_features)

scaler_X.fit(X_train_2d)
X_dev_scaled = scaler_X.transform(X_train_2d).reshape(X_train_raw.shape)
X_test_scaled = scaler_X.transform(X_test_2d).reshape(X_test_raw.shape)

scaler_y.fit(y_train_raw.reshape(-1, 1))
y_dev_scaled = scaler_y.transform(y_train_raw.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test_raw.reshape(-1, 1)).flatten()

val_fraction = 0.20
val_index = int(len(X_dev_scaled) * (1 - val_fraction))
X_train_scaled, X_val_scaled = X_dev_scaled[:val_index], X_dev_scaled[val_index:]
y_train_scaled, y_val_scaled = y_dev_scaled[:val_index], y_dev_scaled[val_index:]

print("Dataset shapes after preparation:")
print(f"  X_train: {X_train_scaled.shape}, y_train: {y_train_scaled.shape}")
print(f"  X_val: {X_val_scaled.shape}, y_val: {y_val_scaled.shape}")
print(f"  X_test: {X_test_scaled.shape}, y_test: {y_test_scaled.shape}")
print(f"  Test dates: {dates_test.shape}")

time_steps = X_train_scaled.shape[1]
n_features = X_train_scaled.shape[2]

@keras.utils.register_keras_serializable(package="Custom", name="QuantumKernelLayer")
class QuantumKernelLayer(layers.Layer):
    def __init__(self, n_qubits=4, n_centers=16, **kwargs):
        super(QuantumKernelLayer, self).__init__(**kwargs)
        self.n_qubits = n_qubits
        self.n_centers = n_centers

    def build(self, input_shape):
        self.centers = self.add_weight(
            name="quantum_centers",
            shape=(self.n_centers, self.n_qubits),
            initializer=keras.initializers.RandomUniform(minval=-np.pi, maxval=np.pi),
            trainable=True,
            dtype=tf.float32
        )
        super(QuantumKernelLayer, self).build(input_shape)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.n_centers)

    def call(self, inputs):
        inputs = tf.cast(inputs, tf.float32)
        batch_size = tf.shape(inputs)[0]
        features_batch = tf.concat([tf.sin(inputs), tf.cos(inputs)], axis=1)
        features_centers = tf.concat([tf.sin(self.centers), tf.cos(self.centers)], axis=1)

        numerator = tf.matmul(features_batch, features_centers, transpose_b=True)
        norms_batch = tf.norm(features_batch, axis=1, keepdims=True) + 1e-8
        norms_centers = tf.norm(features_centers, axis=1, keepdims=True) + 1e-8
        denom = norms_batch * tf.transpose(norms_centers)

        kernel_matrix = numerator / denom
        out = tf.reshape(tf.cast(kernel_matrix, tf.float32), (batch_size, self.n_centers))
        out.set_shape([inputs.shape[0], self.n_centers])
        return out

    def get_config(self):
        cfg = super(QuantumKernelLayer, self).get_config()
        cfg.update({"n_qubits": self.n_qubits, "n_centers": self.n_centers})
        return cfg

def create_qsvm_model(time_steps, n_features, n_qubits, n_centers=16):
    inputs = layers.Input(shape=(time_steps, n_features), dtype=tf.float32)
    x = layers.GRU(64, return_sequences=False, dtype=tf.float32)(inputs)
    x = layers.BatchNormalization(dtype=tf.float32)(x)
    x = layers.Dropout(0.3)(x)

    classical_branch = layers.Dense(16, activation='relu', dtype=tf.float32)(x)
    classical_branch = layers.BatchNormalization(dtype=tf.float32)(classical_branch)
    classical_branch = layers.Dropout(0.3)(classical_branch)

    quantum_input = layers.Dense(n_qubits, activation='tanh', dtype=tf.float32)(x)
    quantum_branch = QuantumKernelLayer(n_qubits=n_qubits, n_centers=n_centers)(quantum_input)

    merged = layers.Concatenate()([classical_branch, quantum_branch])

    x = layers.Dense(64, activation='relu', dtype=tf.float32)(merged)
    x = layers.BatchNormalization(dtype=tf.float32)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(32, activation='relu', dtype=tf.float32)(x)
    x = layers.BatchNormalization(dtype=tf.float32)(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(1, activation='linear', dtype=tf.float32)(x)
    return keras.Model(inputs=inputs, outputs=outputs)

model = create_qsvm_model(time_steps, n_features, n_qubits=4, n_centers=16)

model.compile(
    optimizer=keras.optimizers.Nadam(learning_rate=0.001, clipnorm=1.0),
    loss='mse',
    metrics=['mae']
)

print(model.summary())

class CustomModelCheckpoint(keras.callbacks.Callback):
    def __init__(self, filepath, monitor='val_loss', mode='min'):
        super(CustomModelCheckpoint, self).__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.best = np.inf if mode == 'min' else -np.inf
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        if self.mode == 'min':
            is_best = current < self.best
        else:
            is_best = current > self.best

        if is_best:
            print(f"\nEpoch {epoch + 1}: {self.monitor} improved ({self.best:.6f} -> {current:.6f}). Saved best model to {self.filepath}")
            self.best = current
            self.best_epoch = epoch + 1
            self.model.save(self.filepath, overwrite=True, include_optimizer=True)
        else:
            print(f" (Epoch {epoch + 1}: {self.monitor} did not improve. Best: {self.best:.6f} at epoch {self.best_epoch})")

os.makedirs('models', exist_ok=True)
model_checkpoint = CustomModelCheckpoint(
    filepath='../models/QSVM_regression.keras',
    monitor='val_loss',
    mode='min'
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=20,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train_scaled, y_train_scaled,
    epochs=400,
    batch_size=16,
    validation_data=(X_val_scaled, y_val_scaled),
    callbacks=[early_stopping, reduce_lr, model_checkpoint],
    verbose=1
)

y_pred_scaled = model.predict(X_test_scaled, verbose=0).flatten()

y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_test_original = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1)).flatten()

mse = mean_squared_error(y_test_original, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_original, y_pred)
r2 = r2_score(y_test_original, y_pred)
mape = np.mean(np.abs((y_test_original - y_pred) / y_test_original)) * 100

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"MAPE: {mape:.2f}%")
print(f"R2 Score: {r2:.4f} ({r2*100:.2f}%)")
